# X-OPT - STRUCTURAL INTERPRETABILITY BENCHMARK

This notebook benchmarks the post-hoc structural interpretability pipeline over the OR-Library p-median instances. For each instance, the notebook performs the following steps:

1. run the metaheuristic;
2. build the long-term memory (LTM);
3. keep the top 5% best LTM solutions;
4. build the weighted co-occurrence graph;
5. extract communities, Max-p-Cut, k-core, and densest subgraph;
6. compute the requested interpretability metrics;
7. display and save the final CSV table.

The notebook also stores:

- the metaheuristic runtime from step 1; and
- the structure-extraction runtime from step 5,

so the runtime cost of interpretability can be evaluated directly.

### SETUP

Adding a new path to the Python interpreter:

In [1]:
import sys

from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "instances").exists() and \
           (candidate / "pymedian" ).exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing 'instances' and 'pymedian'."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


Importing the libraries:

In [2]:
from __future__ import annotations

import re
import math
import pymedian

import numpy    as np
import pandas   as pd
import networkx as nx

from tqdm.auto           import tqdm
from networkx.algorithms import community
from itertools           import combinations
from time                import perf_counter

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

Defining helper functions:

In [3]:
def instance_sort_key(pathlike: str | Path) -> tuple[int, str]:
    stem  = Path     (pathlike ).stem
    match = re.search(r"(\d+)$", stem)

    if match is None:
        return (10**9, stem)

    return (int(match.group(1)), stem)


def canonical_instance_name(instance_name: str) -> str:
    instance_name = instance_name.strip()

    if instance_name.endswith(".txt"):
        return instance_name

    return f"{instance_name}.txt"


def list_orlibrary_instances(instances_dir: Path) -> list[str]:
    return sorted(
        [
            path.name
            for path in instances_dir.glob("pmed*.txt")
            if  path.name != "pmedopt.txt"
        ],
        key=instance_sort_key,
    )


def read_instance_metadata(instance_path: Path) -> dict[str, int]:
    header = instance_path.read_text().splitlines()[0].split()

    if len(header) < 3:
        raise ValueError(
            f"Could not parse instance header: {instance_path}"
        )

    return {
        "n": int(header[0]),
        "p": int(header[2]),
    }


def load_best_known_costs(pmedopt_path: Path) -> pd.DataFrame:
    rows = []

    for raw_line in pmedopt_path.read_text().splitlines()[1:]:
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()

        rows.append(
            {
                "instance_id"     : parts[0].strip(),
                "best_known_cost" : float(parts[1] ),
            }
        )

    df = pd.DataFrame(rows)

    df["instance_order"] = df["instance_id"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        df.sort_values(["instance_order", "instance_id"])
          .drop       (columns="instance_order"         )
          .reset_index(drop   =True)
    )


def compute_gap_percent(
    cost            : float | None,
    best_known_cost : float | None,
) -> float:
    if cost is None or pd.isna(cost):
        return np.nan

    if best_known_cost  is None or \
       pd.isna(best_known_cost) or \
       float  (best_known_cost) == 0.0:
        return np.nan

    return 100.0 * (float(cost) - float(best_known_cost)) / float(best_known_cost)

### EXPERIMENT CONFIGURATION

In [4]:
INSTANCES_DIR = PROJECT_ROOT  / "instances"
PMEDOPT_PATH  = INSTANCES_DIR / "pmedopt.txt"

OUTPUT_DIR   = PROJECT_ROOT / "notebooks" / "experiments_sbpo" / "artifacts"
RESULTS_CSV  = OUTPUT_DIR   / "structural_interpretability_benchmark.csv"
FAILURES_CSV = OUTPUT_DIR   / "structural_interpretability_benchmark_failures.csv"

DEFAULT_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES         = DEFAULT_INSTANCE_NAMES


RESTARTS = 8
MAX_ITER = 25
FACTOR   = 1
DETAILS_FORMAT = "binary"

TOP_FRACTION = 0.05

GLOBAL_SEED        = 42
MAX_P_CUT_RESTARTS = 30
MAX_P_CUT_MAX_ITER = 2000


SAVE_RESULTS_CSV  = True
SAVE_FAILURES_CSV = True

BEST_KNOWN_COSTS_DF = load_best_known_costs(PMEDOPT_PATH)
BEST_KNOWN_COSTS    = BEST_KNOWN_COSTS_DF.set_index("instance_id")["best_known_cost"].to_dict()

Verifying the hyperparameters:

In [5]:
print(f"Project root        : {PROJECT_ROOT }")
print(f"Instances folder    : {INSTANCES_DIR}")
print(f"Results CSV         : {RESULTS_CSV  }")

Project root        : /home/rei-luisinho/xopt
Instances folder    : /home/rei-luisinho/xopt/instances
Results CSV         : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/structural_interpretability_benchmark.csv


In [6]:
print(f"Number of instances            : {len(INSTANCE_NAMES)}")
print(f"Top fraction kept from the LTM : {TOP_FRACTION   :.0%}")
print(f"Metaheuristic parameters       : restarts={RESTARTS       :2d}, max_iter={MAX_ITER       :4d}, factor={FACTOR     }")
print(f"Max-p-Cut parameters           : restarts={MAX_P_CUT_RESTARTS}, max_iter={MAX_P_CUT_MAX_ITER}, seed  ={GLOBAL_SEED}")

Number of instances            : 40
Top fraction kept from the LTM : 5%
Metaheuristic parameters       : restarts= 8, max_iter=  25, factor=1
Max-p-Cut parameters           : restarts=30, max_iter=2000, seed  =42


In [7]:
selected_instances_df = pd.DataFrame(
    [
        {
            "instance"        :      canonical_instance_name(instance_name)      ,
            "instance_id"     : Path(canonical_instance_name(instance_name)).stem,
            "best_known_cost" : BEST_KNOWN_COSTS.get(
                Path(canonical_instance_name(instance_name)).stem, np.nan,
            ),
            **read_instance_metadata(INSTANCES_DIR / canonical_instance_name(instance_name)),
        }
        for instance_name in INSTANCE_NAMES
    ]
)

selected_instances_df["instance_order"] = selected_instances_df["instance"].map(
    lambda value: instance_sort_key(value)[0]
)

selected_instances_df = (
    selected_instances_df.sort_values(["instance_order", "instance"])
                         .drop       (columns="instance_order"      )
                         .reset_index(drop   =True)
)

display(selected_instances_df.head(10))

,instance,instance_id,best_known_cost,n,p
0,pmed1.txt,pmed1,5819.0,100,5
1,pmed2.txt,pmed2,4093.0,100,10
2,pmed3.txt,pmed3,4250.0,100,10
3,pmed4.txt,pmed4,3034.0,100,20
4,pmed5.txt,pmed5,1355.0,100,33
5,pmed6.txt,pmed6,7824.0,200,5
6,pmed7.txt,pmed7,5631.0,200,10
7,pmed8.txt,pmed8,4445.0,200,20
8,pmed9.txt,pmed9,2734.0,200,40
9,pmed10.txt,pmed10,1255.0,200,67


### METRIC DEFINITIONS

Let:

- $B$ be the set of facilities in the best solution returned by the metaheuristic;
- $G_{top} = (V, E_{top})$ be the weighted co-occurrence graph built from the top 5% LTM solutions;
- $A$ be its weighted adjacency matrix;
- $C_1, \ldots, C_k$ be the communities found in $G_{top}$;
- $D \subseteq V$ be the node set of the densest subgraph;
- $K_{\max}$ be the highest k-core.

The benchmark uses the following definitions.

#### CO-OCCURRENCE GRAPH

In [8]:
def build_top_ltm(
    long_term_memory : list[dict[str, object]],
    top_fraction     : float,
) -> tuple[list[dict[str, object]], np.ndarray, np.ndarray]:
    if not long_term_memory:
        raise ValueError("long_term_memory is empty.")

    top_solution_count = max(
        1, int(np.ceil(len(long_term_memory) * top_fraction))
    )

    analysis_ltm = sorted(
        long_term_memory, key=lambda sol: float(sol["cost"])
    )[:top_solution_count]

    matrix = np.vstack(
        [
            np.asarray(
                sol["facilities"], dtype=np.int8
            )
            for sol in analysis_ltm
        ]
    )

    costs = np.asarray(
        [
            float(sol["cost"])
            for sol in analysis_ltm
        ],
        dtype=float,
    )

    return analysis_ltm, matrix, costs


def build_cooccurrence_matrix(matrix: np.ndarray) -> np.ndarray:
    adjacency = np.asarray(
        matrix.T @ matrix, dtype=np.int64
    )

    np.fill_diagonal(adjacency, 0)

    return adjacency


def build_weighted_graph(adjacency: np.ndarray) -> nx.Graph:
    n = adjacency.shape[0]

    graph = nx.Graph()
    graph.add_nodes_from(range(n))

    rows, cols = np.where(np.triu(adjacency, k=1) > 0)

    for i, j in zip(rows.tolist(), cols.tolist()):
        graph.add_edge(
            int(i), int(j), weight=float(adjacency[i, j])
        )

    return graph


def build_unweighted_graph(adjacency: np.ndarray) -> nx.Graph:
    n = adjacency.shape[0]

    graph = nx.Graph()
    graph.add_nodes_from(range(n))

    rows, cols = np.where(np.triu(adjacency, k=1) > 0)

    graph.add_edges_from(
        (int(i), int(j))
        for i, j in zip(rows.tolist(), cols.tolist())
    )

    return graph


def total_edge_weight(adjacency: np.ndarray) -> float:
    return float(np.triu(adjacency, k=1).sum())


def total_edge_count(adjacency: np.ndarray) -> int:
    return int(np.count_nonzero(np.triu(adjacency, k=1) > 0))

#### COMMUNITIES

- **Weighted modularity**

It's the most natural metric if you're using community detection in a weighted graph; the higher the $Q$ value, the more coherent the partition.

$$
Q_w = \frac{1}{2m}\sum_{i,j}\left(A_{ij} - \frac{k_i^w k_j^w}{2m}\right)\mathbf{1}[c_i = c_j],
$$

where $k_i^w = \sum_j A_{ij}$ and $2m = \sum_{i,j} A_{ij}$.

- **Coverage**

Fraction of the total weight of the edges that remains within the communities.

$$
\mathrm{Cov} = \frac{\sum_{g=1}^{k}\sum_{i<j,\ i,j\in C_g} A_{ij}}{\sum_{i<j} A_{ij}}
$$

- **Best-facility concentration**

If $B$ is the set of the $p$ facilities in the best solution, then:

$$
\mathrm{BFC} = \sum_{g=1}^{k}\left(\frac{|B \cap C_g|}{|B|}\right)^2
$$

- close to $1$: the best facilities are concentrated in few communities;
- lower values: the best facilities are spread across many communities.

- **Best-facility purity**

This is the percentage of communities with size at least $2$ that contain at least one best facility:

$$
\mathrm{BFP} =
\frac{\left|\left\{g : |C_g| \ge 2,\ B \cap C_g \neq \emptyset\right\}\right|}
     {\left|\left\{g : |C_g| \ge 2\right\}\right|}
$$

In [9]:
def community_internal_weight(
    adjacency: np.ndarray, nodes: set[int] | frozenset[int]
) -> float:
    node_array = np.array(sorted(nodes), dtype=int)

    if node_array.size <= 1:
        return 0.0

    subgraph = adjacency[np.ix_(node_array, node_array)]

    return float(np.triu(subgraph, k=1).sum())


def induced_edge_count(
    adjacency: np.ndarray, nodes: set[int]
) -> int:
    if len(nodes) <= 1:
        return 0

    node_list = sorted(nodes)
    subgraph  = adjacency[np.ix_(node_list, node_list)]

    return int(np.count_nonzero(np.triu(subgraph, k=1) > 0))


def weighted_coverage(
    adjacency         : np.ndarray,
    communities_found : list[set[int] | frozenset[int]],
) -> float:
    total_weight = total_edge_weight(adjacency)

    if total_weight <= 0.0:
        return 0.0

    internal_weight = sum(
        community_internal_weight(
            adjacency, community_nodes
        )
        for community_nodes in communities_found
    )

    return internal_weight / total_weight


def best_facility_concentration(
    communities_found : list[set[int] | frozenset[int]],
    best_set          : set [int],
) -> float:
    if not best_set:
        return 0.0

    best_size = len(best_set)

    return float(
        sum(
            (len(set(community_nodes).intersection(best_set)) / best_size) ** 2
            for community_nodes in communities_found
        )
    )


def best_facility_purity(
    communities_found : list[set[int] | frozenset[int]],
    best_set          : set [int],
) -> float:
    eligible_communities = [
        set(community_nodes)
        for community_nodes in communities_found
        if len(community_nodes) >= 2
    ]

    if not eligible_communities:
        return 0.0

    communities_with_best = sum(
        1
        for community_nodes in eligible_communities
        if  community_nodes.intersection(best_set)
    )

    return communities_with_best / len(eligible_communities)


def community_metrics(
    graph     : nx.Graph  ,
    adjacency : np.ndarray,
    best_set  : set[int]  ,
) -> dict[str, float | int]:
    if graph.number_of_edges() == 0:
        communities_found   = [
            frozenset([node]) for node in graph.nodes
        ]
        weighted_modularity = 0.0
    else:
        communities_found   = list(
            community.greedy_modularity_communities(graph, weight="weight")
        )
        weighted_modularity = float(
            nx.community.modularity(graph, communities_found, weight="weight")
        )

    return {
        "communities_count"               : len(communities_found),
        "communities_weighted_modularity" : weighted_modularity   ,
        "communities_coverage"            : weighted_coverage(adjacency, communities_found),

        "communities_best_facility_concentration" : best_facility_concentration(communities_found, best_set),
        "communities_best_facility_purity"        : best_facility_purity       (communities_found, best_set),
    }

#### MAX-p-CUT

- **Fraction cut**

$$
\mathrm{FC} = \frac{W_{\mathrm{cut}}}{W_{\mathrm{cut}} + W_{\mathrm{int}}}
$$

- **Best-facility separation**

$$
\mathrm{BFS} =
\frac{1}{\binom{|B|}{2}}
\sum_{\{i,j\}\subseteq B}\mathbf{1}[\ell_i \ne \ell_j]
$$

where $\ell_i$ is the Max-p-Cut label assigned to facility $i$.

In [10]:
def _random_labels(n: int, p: int, rng: np.random.Generator) -> np.ndarray:
    labels = rng.integers(0, p, size=n)

    if n >= p:
        permutation             = rng.permutation(n)
        labels[permutation[:p]] = np .arange     (p)

    return labels


def _group_sums(A: np.ndarray, labels: np.ndarray, p: int) -> np.ndarray:
    sums = np.zeros((A.shape[0], p), dtype=float)

    for group_id in range(p):
        members = labels == group_id

        if np.any(members):
            sums[:, group_id] = A[:, members].sum(axis=1)

    return sums


def max_p_cut_local_search(
    A: np.ndarray,
    p: int       ,
    n_restarts: int = 30  ,
    max_iter  : int = 2000,
    seed      : int = 42  ,
) -> tuple[np.ndarray, float, float]:
    A = np.asarray(A, dtype=float)

    n = A.shape[0]
    p = max(1, min(int(p), n))

    total_weight = float(np.triu(A, 1).sum())

    if total_weight <= 1e-12:
        labels = np.arange(n, dtype=int) % p

        return labels, 0.0, 0.0

    rng = np.random.default_rng(seed)

    best_cut      = -np.inf
    best_internal =  np.inf
    best_labels   = None

    for _ in range(max(1, int(n_restarts))):
        labels = _random_labels(n, p, rng)
        sums   = _group_sums   (A, labels, p)

        internal = float(np.triu(A * (labels[:, None] == labels[None, :]), 1).sum())
        cut      = total_weight - internal

        for _ in range(max(1, int(max_iter))):
            improved = False

            for node in rng.permutation(n):
                old_group = labels[node]

                gains = sums[node, old_group] - sums[node, :]
                gains[old_group] = -np.inf

                new_group = int  (np.argmax(gains))
                gain      = float(gains[new_group])

                if gain > 1e-12:
                    labels[node] = new_group
                    cut += gain

                    sums[:, old_group] -= A[:, node]
                    sums[:, new_group] += A[:, node]

                    improved = True

            if not improved:
                break

        internal = total_weight - cut

        if cut > best_cut + 1e-12:
            best_cut      = float(cut     )
            best_internal = float(internal)
            best_labels   = labels.copy()

    return best_labels, best_cut, best_internal


def best_facility_separation(
    labels: np.ndarray, best_set: set[int]
) -> float:
    best_nodes = sorted(best_set)

    if len(best_nodes) <= 1:
        return 1.0

    separated_pairs = sum(
        labels[i] != labels[j]
        for i, j in combinations(best_nodes, 2)
    )

    return separated_pairs / math.comb(len(best_nodes), 2)

#### K-CORE

- **Core mass**

$$
\mathrm{CM} = \frac{|K_{\max}|}{|V|}
$$

- **Best-core recall**

$$
\mathrm{BCR} = \frac{|B \cap K_{\max}|}{|B|}
$$

- **Best-core precision**

$$
\mathrm{BCP} = \frac{|B \cap K_{\max}|}{|K_{\max}|}
$$

In [11]:
def k_core_metrics(
    graph    : nx.Graph,
    n        : int     ,
    best_set : set[int],
) -> dict[str, float | int]:
    core_numbers   = nx.core_number(graph)

    max_core_level = max(
        core_numbers.values()
    ) if core_numbers else 0

    highest_core_nodes = {
        node
        for node, core_level in core_numbers.items()
        if  core_level == max_core_level
    }

    overlap = highest_core_nodes.intersection(best_set)

    return {
        "k_core_max_level" : int(max_core_level    ),
        "k_core_core_size" : len(highest_core_nodes),
        "k_core_core_mass" : len(highest_core_nodes) / max(1, n),

        "k_core_best_core_recall"    : len(overlap) / max(1, len(best_set          )),
        "k_core_best_core_precision" : len(overlap) / max(1, len(highest_core_nodes)),
    }

#### DENSEST SUBGRAPH

- **Average internal weighted degree**

$$
\mathrm{AIWD} = \frac{1}{|D|}\sum_{i\in D}\sum_{j\in D} A_{ij}
$$

- **Best overlap**

$$
\mathrm{BO} = \frac{|D \cap B|}{|B|}
$$

- **Original-edge percentage**

$$
\mathrm{OEP} = \frac{|E(D)|}{|E_{top}|}
$$

where $E(D)$ is the set of positive-weight edges induced by $D$ inside the original co-occurrence graph.

In [12]:
def densest_subgraph_greedy(
    adjacency : np.ndarray,
    min_size  : int = 3   ,
) -> tuple[set[int], float]:
    n = adjacency.shape[0]

    remaining     = set(range(n))
    best_subgraph = set(range(n)) if n and n < min_size else set()
    best_density  = -np.inf

    while len(remaining) >= max(1, min_size):
        nodes    = sorted   (remaining)
        subgraph = adjacency[np.ix_(nodes, nodes)]

        current_weight  = float(np.triu(subgraph, k=1).sum())
        current_density = current_weight / max(1, len(remaining))

        if current_density > best_density:
            best_density  = current_density
            best_subgraph = remaining.copy()

        min_pos = int(
            np.argmin(subgraph.sum(axis=1))
        )
        remaining.remove(nodes[min_pos])

    if not best_subgraph and n:
        best_subgraph = set(range(min(n, max(1, min_size))))
        best_density  = 0.0

    return best_subgraph, max(0.0, float(best_density))


def average_internal_weighted_degree(
    adjacency : np.ndarray,
    nodes     : set[int]  ,
) -> float:
    if not nodes:
        return 0.0

    node_list        = sorted(nodes)
    subgraph         = adjacency[np.ix_(node_list, node_list)]
    weighted_degrees = subgraph.sum(axis=1)

    return float(np.mean(weighted_degrees))


def jaccard_overlap(
    set_a: set[int],
    set_b: set[int],
) -> float:
    union = set_a.union(set_b)

    if not union:
        return 1.0

    return len(set_a.intersection(set_b)) / len(union)


def densest_subgraph_edge_percentage(
    adjacency     : np.ndarray,
    densest_nodes : set[int]  ,
) -> float:
    total_edges = total_edge_count(adjacency)

    if total_edges <= 0:
        return 0.0

    return induced_edge_count(adjacency, densest_nodes) / total_edges


def densest_subgraph_metrics(
    adjacency : np.ndarray,
    best_set  : set[int]  ,
) -> dict[str, float | int]:
    densest_nodes, _ = densest_subgraph_greedy(
        adjacency, min_size=3
    )

    return {
        "densest_subgraph_size" : len(densest_nodes),

        "densest_subgraph_average_internal_weighted_degree" : average_internal_weighted_degree(adjacency    , densest_nodes),
        "densest_subgraph_edge_percentage"                  : densest_subgraph_edge_percentage(adjacency    , densest_nodes),
        "densest_subgraph_best_overlap"                     : jaccard_overlap                 (densest_nodes, best_set     ),
    }

#### IMPLEMENTATION

In [13]:
def run_single_instance_analysis(
    instance_name: str,
    *,
    restarts     : int  ,
    max_iter     : int  ,
    factor       : int  ,
    top_fraction : float,

    details_format     : str = "binary",
    max_p_cut_restarts : int = 30  ,
    max_p_cut_max_iter : int = 2000,
    global_seed        : int = 42  ,
) -> dict[str, object]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    row = {
        "instance"    : instance_name,
        "instance_id" : instance_id  ,
        "n"           : np.nan,
        "p"           : np.nan,

        "best_known_cost"                      : BEST_KNOWN_COSTS.get(instance_id, np.nan),
        "best_cost"                            : np.nan,
        "gap_percent"                          : np.nan,
        "memory_size"                          : np.nan,
        "top_solution_count"                   : np.nan,
        "top_cost_cutoff"                      : np.nan,
        "cooccurrence_edges"                   : np.nan,
        "cooccurrence_total_weight"            : np.nan,
        "metaheuristic_runtime_seconds"        : np.nan,
        "structure_extraction_runtime_seconds" : np.nan,
        "total_pipeline_runtime_seconds"       : np.nan,

        "communities_count"                       : np.nan,
        "communities_weighted_modularity"         : np.nan,
        "communities_coverage"                    : np.nan,
        "communities_best_facility_concentration" : np.nan,
        "communities_best_facility_purity"        : np.nan,

        "max_p_cut_fraction_cut"             : np.nan,
        "max_p_cut_best_facility_separation" : np.nan,

        "k_core_max_level"           : np.nan,
        "k_core_core_size"           : np.nan,
        "k_core_core_mass"           : np.nan,
        "k_core_best_core_recall"    : np.nan,
        "k_core_best_core_precision" : np.nan,

        "densest_subgraph_size"                             : np.nan,
        "densest_subgraph_average_internal_weighted_degree" : np.nan,
        "densest_subgraph_edge_percentage"                  : np.nan,
        "densest_subgraph_best_overlap"                     : np.nan,

        "status"        : "error",
        "error_message" : None   ,
    }

    if not instance_path.exists():
        row["error_message"] = f"Instance not found: {instance_path}"

        return row

    metadata = read_instance_metadata(instance_path)
    row["n"] = metadata["n"]
    row["p"] = metadata["p"]

    overall_started_at = perf_counter()

    try:
        meta_started_at = perf_counter()

        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      =restarts      ,
            max_iter      =max_iter      ,
            factor        =factor        ,
            details_format=details_format,
        )

        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        long_term_memory = details["long_term_memory"]
        if not long_term_memory:
            raise ValueError("long_term_memory is empty.")

        analysis_ltm, matrix, costs = build_top_ltm            (long_term_memory, top_fraction)
        adjacency                   = build_cooccurrence_matrix(matrix)

        weighted_graph   = build_weighted_graph  (adjacency)
        unweighted_graph = build_unweighted_graph(adjacency)

        best_facilities = tuple(
            sorted(
                int(value) - 1
                for value in summary["tspmed_facilities"]
            )
        )
        best_set        = set(best_facilities)

        structure_started_at = perf_counter()

        community_stats = community_metrics(weighted_graph, adjacency, best_set)

        labels_maxcut, cut_weight, internal_weight = max_p_cut_local_search(
            adjacency        ,
            int(summary["p"]),
            n_restarts=max_p_cut_restarts,
            max_iter  =max_p_cut_max_iter,
            seed      =global_seed       ,
        )

        max_p_cut_stats = {
            "max_p_cut_fraction_cut"             : cut_weight / max(1e-12, cut_weight + internal_weight),
            "max_p_cut_best_facility_separation" : best_facility_separation(labels_maxcut, best_set    ),
        }

        k_core_stats = k_core_metrics(
            unweighted_graph ,
            int(summary["n"]),
            best_set         ,
        )

        densest_stats = densest_subgraph_metrics(adjacency, best_set)

        structure_extraction_runtime_seconds = perf_counter() - structure_started_at
        total_pipeline_runtime_seconds       = perf_counter() - overall_started_at

        row.update(
            {
                "n"           : int  (summary["n"          ]),
                "p"           : int  (summary["p"          ]),
                "best_cost"   : float(summary["tspmed_cost"]),
                "gap_percent" : compute_gap_percent(
                    summary["tspmed_cost"    ],
                    row    ["best_known_cost"],
                ),

                "memory_size"               : len  (long_term_memory                ),
                "top_solution_count"        : len  (analysis_ltm                    ),
                "top_cost_cutoff"           : float(costs.max()                     ),
                "cooccurrence_edges"        : int  (weighted_graph.number_of_edges()),
                "cooccurrence_total_weight" : total_edge_weight(adjacency           ),

                "metaheuristic_runtime_seconds"        : metaheuristic_runtime_seconds       ,
                "structure_extraction_runtime_seconds" : structure_extraction_runtime_seconds,
                "total_pipeline_runtime_seconds"       : total_pipeline_runtime_seconds      ,

                **community_stats,
                **max_p_cut_stats,
                **k_core_stats   ,
                **densest_stats  ,

                "status"        : "ok",
                "error_message" : None,
            }
        )
    except Exception as exc:
        row["total_pipeline_runtime_seconds"] = perf_counter() - overall_started_at
        row["error_message"                 ] = f"{type(exc).__name__}: {exc}"

    return row


def run_benchmark(
    instance_names: list[str],
    *,
    restarts     : int  ,
    max_iter     : int  ,
    factor       : int  ,
    top_fraction : float,

    details_format     : str = "binary",
    max_p_cut_restarts : int = 30  ,
    max_p_cut_max_iter : int = 2000,
    global_seed        : int = 42  ,
) -> pd.DataFrame:
    if not instance_names:
        raise ValueError(
            "The benchmark requires at least one instance."
        )

    rows = []

    for instance_name in tqdm(
        instance_names,
        total        =len(instance_names)   ,
        desc         ="Structural benchmark",
        dynamic_ncols=True                  ,
    ):
        rows.append(
            run_single_instance_analysis(
                instance_name,
                restarts          =restarts          ,
                max_iter          =max_iter          ,
                factor            =factor            ,
                top_fraction      =top_fraction      ,
                details_format    =details_format    ,
                max_p_cut_restarts=max_p_cut_restarts,
                max_p_cut_max_iter=max_p_cut_max_iter,
                global_seed       =global_seed       ,
            )
        )

    results_df = pd.DataFrame(rows)

    results_df["interpretability_overhead_ratio"] = np.where(
        results_df["metaheuristic_runtime_seconds"       ] > 0,
        results_df["structure_extraction_runtime_seconds"] / 
        results_df["metaheuristic_runtime_seconds"       ],
        np.nan,
    )

    results_df["instance_order"] = results_df["instance"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        results_df.sort_values(["instance_order", "instance"])
                  .drop       (columns="instance_order"      )
                  .reset_index(drop   =True)
    )

### APPLY

In [14]:
results_df = run_benchmark(
    INSTANCE_NAMES,
    restarts          =RESTARTS          ,
    max_iter          =MAX_ITER          ,
    factor            =FACTOR            ,
    top_fraction      =TOP_FRACTION      ,
    details_format    =DETAILS_FORMAT    ,
    max_p_cut_restarts=MAX_P_CUT_RESTARTS,
    max_p_cut_max_iter=MAX_P_CUT_MAX_ITER,
    global_seed       =GLOBAL_SEED       ,
)

final_column_order = [
    "instance"   ,
    "instance_id",
    "n",
    "p",
    "best_known_cost",
    "best_cost"      ,
    "gap_percent"    ,
    "memory_size"    ,

    "top_solution_count"       ,
    "top_cost_cutoff"          ,
    "cooccurrence_edges"       ,
    "cooccurrence_total_weight",

    "metaheuristic_runtime_seconds"       ,
    "structure_extraction_runtime_seconds",
    "interpretability_overhead_ratio"     ,
    "total_pipeline_runtime_seconds"      ,

    "communities_count"                      ,
    "communities_weighted_modularity"        ,
    "communities_coverage"                   ,
    "communities_best_facility_concentration",
    "communities_best_facility_purity"       ,

    "max_p_cut_fraction_cut"            ,
    "max_p_cut_best_facility_separation",

    "k_core_max_level"          ,
    "k_core_core_size"          ,
    "k_core_core_mass"          ,
    "k_core_best_core_recall"   ,
    "k_core_best_core_precision",

    "densest_subgraph_size"                            ,
    "densest_subgraph_average_internal_weighted_degree",
    "densest_subgraph_edge_percentage"                 ,
    "densest_subgraph_best_overlap"                    ,

    "status"       ,
    "error_message",
]

final_results_df = results_df[final_column_order].copy()

display(final_results_df)

Structural benchmark:   0%|          | 0/40 [00:00<?, ?it/s]

,instance,instance_id,n,p,best_known_cost,best_cost,gap_percent,memory_size,top_solution_count,top_cost_cutoff,cooccurrence_edges,cooccurrence_total_weight,metaheuristic_runtime_seconds,structure_extraction_runtime_seconds,interpretability_overhead_ratio,total_pipeline_runtime_seconds,communities_count,communities_weighted_modularity,communities_coverage,communities_best_facility_concentration,communities_best_facility_purity,max_p_cut_fraction_cut,max_p_cut_best_facility_separation,k_core_max_level,k_core_core_size,k_core_core_mass,k_core_best_core_recall,k_core_best_core_precision,densest_subgraph_size,densest_subgraph_average_internal_weighted_degree,densest_subgraph_edge_percentage,densest_subgraph_best_overlap,status,error_message
0,pmed1.txt,pmed1,100,5,5819.0,5819.0,0.000000,119,6,5845.0,27,60.0,0.032389,0.038264,1.181392,0.072166,92,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,5,7,0.070000,1.000000,0.714286,4,15.000000,0.222222,0.800000,ok,None
1,pmed2.txt,pmed2,100,10,4093.0,4093.0,0.000000,137,7,4100.0,83,315.0,0.067819,0.034640,0.510775,0.103195,87,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,10,13,0.130000,1.000000,0.769231,11,49.090909,0.650602,0.909091,ok,None
2,pmed3.txt,pmed3,100,10,4250.0,4250.0,0.000000,89,5,4253.0,81,225.0,0.049306,0.033763,0.684768,0.083642,87,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,9,14,0.140000,1.000000,0.714286,9,38.222222,0.444444,0.900000,ok,None
3,pmed4.txt,pmed4,100,20,3034.0,3034.0,0.000000,111,6,3034.0,249,1140.0,0.090150,0.036769,0.407868,0.127990,78,0.000000e+00,1.000000,0.815000,1.000000,1.000000,0.989474,20,23,0.230000,0.900000,0.782609,21,101.333333,0.839357,0.708333,ok,None
4,pmed5.txt,pmed5,100,33,1355.0,1355.0,0.000000,117,6,1355.0,660,3168.0,0.130241,0.047008,0.360929,0.178655,65,5.802979e-04,0.904672,0.632691,1.000000,1.000000,0.992424,34,36,0.360000,0.848485,0.777778,35,173.771429,0.898485,0.700000,ok,None
5,pmed6.txt,pmed6,200,5,7824.0,7824.0,0.000000,243,13,7864.0,35,130.0,0.191009,0.068823,0.360314,0.261888,192,1.349112e-02,0.523077,0.520000,1.000000,1.000000,1.000000,6,9,0.045000,1.000000,0.555556,9,28.000000,0.885714,0.555556,ok,None
6,pmed7.txt,pmed7,200,10,5631.0,5631.0,0.000000,111,6,5641.0,83,270.0,0.349101,0.070916,0.203139,0.421172,188,8.888889e-03,0.777778,1.000000,0.500000,1.000000,1.000000,10,14,0.070000,1.000000,0.714286,11,39.636364,0.650602,0.750000,ok,None
7,pmed8.txt,pmed8,200,20,4445.0,4445.0,0.000000,88,5,4446.0,248,950.0,0.854952,0.077298,0.090412,0.933479,178,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,20,22,0.110000,1.000000,0.909091,20,85.600000,0.766129,1.000000,ok,None
8,pmed9.txt,pmed9,200,40,2734.0,2738.0,0.146306,148,8,2738.0,1053,6240.0,1.002148,0.104507,0.104283,1.109435,155,2.636218e-03,0.842949,0.523750,1.000000,1.000000,0.993590,39,47,0.235000,0.875000,0.744681,41,281.951220,0.777778,0.653061,ok,None
9,pmed10.txt,pmed10,200,67,1255.0,1255.0,0.000000,193,10,1255.0,3014,22110.0,1.771658,0.184487,0.104132,1.962509,123,1.860845e-03,0.897512,0.732680,1.000000,1.000000,0.998191,69,73,0.365000,0.880597,0.808219,69,593.855072,0.776709,0.743590,ok,None


In [15]:
success_df = final_results_df.loc[final_results_df["status"] == "ok"].copy()

failure_df = final_results_df.loc[
    final_results_df["status"] != "ok",
    [
        "instance"     ,
        "instance_id"  ,
        "status"       ,
        "error_message",
    ],
].reset_index(drop=True)

print(f"Successful instances : {len(success_df)}")
print(f"Failed instances     : {len(failure_df)}")

print()
if failure_df.empty:
    print  ("No execution failures were recorded.")
else:
    display(failure_df)

Successful instances : 40
Failed instances     : 0

No execution failures were recorded.


Saving the results:

In [16]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR      .mkdir (parents=True, exist_ok=True )
    final_results_df.to_csv(RESULTS_CSV , index   =False)

    print(f"Results saved to  : {RESULTS_CSV}")

if SAVE_FAILURES_CSV and not failure_df.empty:
    OUTPUT_DIR.mkdir (parents=True, exist_ok=True )
    failure_df.to_csv(FAILURES_CSV, index   =False)

    print(f"Failures saved to : {FAILURES_CSV}")

Results saved to  : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/structural_interpretability_benchmark.csv
